In [11]:
import pandas as pd
import icartt
import os
import warnings
import re
from datetime import datetime
import csv
from datetime import datetime, timedelta
from netCDF4 import Dataset
import numpy as np
from scipy import stats
import glob
from math import pi
import ast

In [ ]:
bins = [3.1, 3, 4, 5, 6]
data_strats = [1, 2]

#Set which "MAC_binning_{binned_number}bins_optimized" file to use
binned_number = 3.1
data_strat = 1

STEP 0: Read entire dataset and set output path for split files.

In [15]:


#READ ENTIRE DATASET

if(binned_number == 3.1):
    dataset_path = rf"C:\Users\haika\Desktop\May_Research\MAC Machine Learning Model Code\MAC_DATASET_LLOD_FILTERED.csv"
else:
    dataset_path = rf"C:\Users\haika\Desktop\May_Research\MAC Machine Learning Model Code\binning_datasets\MAC_binning_{binned_number}bins_optimized.csv"

df = pd.read_csv(dataset_path)

#Define output path

if(binned_number == 3.1):
    output_path = rf"C:\Users\haika\Desktop\May_Research\MAC Machine Learning Model Code\split_datasets\data_split_strat{data_strat}\MAC_DATASET_LLOD_FILTERED"
else:
    output_path = rf"C:\Users\haika\Desktop\May_Research\MAC Machine Learning Model Code\split_datasets\data_split_strat{data_strat}\MAC_binning_{binned_number}bins_optimized"


#Define info_lists
kfold_info_path = rf"C:\Users\haika\Desktop\May_Research\MAC Machine Learning Model Code\info_lists\kfold_info_list{data_strat}.csv"
campaign_based_info_path = rf"C:\Users\haika\Desktop\May_Research\MAC Machine Learning Model Code\info_lists\campaign_based_info_list{data_strat}.csv"
day_based_info_path = rf"C:\Users\haika\Desktop\May_Research\MAC Machine Learning Model Code\info_lists\day_based_info_list{data_strat}.csv"
random_based_info_path = rf"C:\Users\haika\Desktop\May_Research\MAC Machine Learning Model Code\info_lists\random_based_info_list{data_strat}.csv"

#Read them as dfs
df_kfold_info = pd.read_csv(kfold_info_path)
df_campaign_based_info = pd.read_csv(campaign_based_info_path)
df_day_based_info = pd.read_csv(day_based_info_path)
df_random_based_info = pd.read_csv(random_based_info_path)

#Columns important for model runs
model_columns = ['N1', 'N2', 'N3', 'N_total', 'V1', 'V2', 'V3', 'V_total', 'SAE', 'AAE', 'SSA_red', 'SSA_blue', 'SSA_green', 'b_scat_red', 'b_scat_blue', 'b_scat_green', 'b_abs_red', 'b_abs_blue', 'b_abs_green', 'MAC_bc']

SPLITTING METHOD 1: 5-Fold Splitting

In [16]:
#SPLIT METHOD 1: Nested Cross-Validation (5-Fold Outer + 3-Fold Inner)
FINAL_OUT_PATH = os.path.join(output_path, "kfold_based_split")
os.makedirs(FINAL_OUT_PATH, exist_ok=True)

print("Starting Nested Cross-Validation dataset splitting...")
print(f"Original dataset shape: {df.shape}")

# Get all unique campaigns in the dataset for reference
all_campaigns_in_data = set(df['Campaign'].unique())
print(f"All campaigns in dataset: {sorted(all_campaigns_in_data)}")

# Define model columns for filtering
model_columns = ['N1', 'N2', 'N3', 'N_total', 'V1', 'V2', 'V3', 'V_total', 'SAE', 'AAE', 'SSA_red', 'SSA_blue', 'SSA_green', 'b_scat_red', 'b_scat_blue', 'b_scat_green', 'b_abs_red', 'b_abs_blue', 'b_abs_green', 'MAC_bc']

# Filter function
def filter_invalid_data(df):
    """Filter out rows with NaN, empty values, -2222, or -9999 in model_columns"""
    if df.shape[0] == 0:
        return df
    
    mask = True
    for col in model_columns:
        if col in df.columns:
            col_mask = (~df[col].isna()) & (df[col] != -2222) & (df[col] != -9999) & (df[col] != '')
            mask = mask & col_mask
        else:
            print(f"Warning: Column '{col}' not found in dataset")
    return df[mask].copy()

# ============================================================================
# STEP 1: Create Hold-out Evaluation Set
# ============================================================================
print("\n" + "="*60)
print("STEP 1: CREATING HOLD-OUT EVALUATION SET")
print("="*60)

# Get evaluation campaigns from CSV
evaluation_campaigns_str = df_kfold_info['evaluation_campaigns'].iloc[0]
evaluation_campaigns = [camp.strip() for camp in evaluation_campaigns_str.split(',') if camp.strip()]

print(f"Evaluation campaigns (hold-out): {evaluation_campaigns}")

# Create evaluation dataset
evaluation_df = df[df['Campaign'].isin(evaluation_campaigns)].copy()
print(f"Evaluation data shape (before filtering): {evaluation_df.shape}")

# Filter evaluation data
evaluation_df_filtered = filter_invalid_data(evaluation_df)
print(f"Evaluation data shape (after filtering): {evaluation_df_filtered.shape}")
print(f"Evaluation rows removed: {evaluation_df.shape[0] - evaluation_df_filtered.shape[0]:,}")

# Save evaluation set
evaluation_dir = os.path.join(FINAL_OUT_PATH, "evaluation")
os.makedirs(evaluation_dir, exist_ok=True)
evaluation_path = os.path.join(evaluation_dir, "evaluation.csv")
evaluation_df_filtered.to_csv(evaluation_path, index=False)
print(f"Saved evaluation.csv to: {evaluation_path}")

# ============================================================================
# STEP 2: Get Remaining Campaigns for Nested CV
# ============================================================================
print("\n" + "="*60)
print("STEP 2: PREPARING CAMPAIGNS FOR NESTED CV")
print("="*60)

# Get remaining campaigns (exclude evaluation campaigns)
remaining_campaigns = list(all_campaigns_in_data - set(evaluation_campaigns))
print(f"Remaining campaigns for nested CV: {sorted(remaining_campaigns)}")
print(f"Number of campaigns for nested CV: {len(remaining_campaigns)}")

if len(remaining_campaigns) < 5:
    print(f"ERROR: Need at least 5 campaigns for 5-fold CV, but only have {len(remaining_campaigns)}")
    raise ValueError("Insufficient campaigns for 5-fold cross-validation")

# ============================================================================
# STEP 3: Create Outer Folds (5-Fold) with Constraint
# ============================================================================
print("\n" + "="*60)
print("STEP 3: CREATING OUTER FOLDS (5-FOLD)")
print("="*60)

import random
random.seed(42)  # For reproducibility

def create_outer_folds_with_constraint(campaigns, n_folds=5):
    """Create outer folds ensuring all campaigns appear in at least one test fold and aiming for 80:20 train:test ratio"""
    
    # Get row counts for each campaign (filtered data)
    campaign_rows = {}
    for campaign in campaigns:
        campaign_data = df[df['Campaign'] == campaign]
        campaign_rows[campaign] = filter_invalid_data(campaign_data).shape[0]
    
    total_rows = sum(campaign_rows.values())
    target_test_rows_per_fold = total_rows * 0.2 / n_folds  # 20% of total divided by number of folds
    
    print(f"Total rows in {len(campaigns)} campaigns: {total_rows:,}")
    print(f"Target test rows per fold: {target_test_rows_per_fold:,.0f}")
    print(f"Campaign row counts: {campaign_rows}")
    
    # Sort campaigns by row count for better distribution
    sorted_campaigns = sorted(campaigns, key=lambda x: campaign_rows[x], reverse=True)
    
    folds = []
    campaigns_used_in_test = set()
    
    for fold_idx in range(n_folds):
        print(f"\n  Creating fold {fold_idx + 1}...")
        
        # For the last fold, ensure all unused campaigns are included
        if fold_idx == n_folds - 1:
            unused_campaigns = set(campaigns) - campaigns_used_in_test
            if unused_campaigns:
                test_campaigns = list(unused_campaigns)
                print(f"    Last fold - including all unused campaigns: {test_campaigns}")
            else:
                # If all campaigns used, create balanced split from remaining options
                test_campaigns = find_best_test_combination(campaigns, campaign_rows, target_test_rows_per_fold, campaigns_used_in_test)
        else:
            # Find best combination for this fold
            test_campaigns = find_best_test_combination(campaigns, campaign_rows, target_test_rows_per_fold, campaigns_used_in_test)
        
        train_campaigns = [c for c in campaigns if c not in test_campaigns]
        
        # Calculate actual ratios
        test_rows = sum(campaign_rows[c] for c in test_campaigns)
        train_rows = sum(campaign_rows[c] for c in train_campaigns)
        total_fold_rows = train_rows + test_rows
        
        test_pct = (test_rows / total_fold_rows * 100) if total_fold_rows > 0 else 0
        train_pct = (train_rows / total_fold_rows * 100) if total_fold_rows > 0 else 0
        
        print(f"    Test campaigns: {test_campaigns} ({test_rows:,} rows, {test_pct:.1f}%)")
        print(f"    Train campaigns: {train_campaigns} ({train_rows:,} rows, {train_pct:.1f}%)")
        
        folds.append({
            'fold': fold_idx + 1,
            'train_campaigns': train_campaigns,
            'test_campaigns': test_campaigns
        })
        
        campaigns_used_in_test.update(test_campaigns)
    
    # Verify all campaigns are used in at least one test fold
    all_test_campaigns = set()
    for fold in folds:
        all_test_campaigns.update(fold['test_campaigns'])
    
    missing_campaigns = set(campaigns) - all_test_campaigns
    if missing_campaigns:
        print(f"WARNING: Campaigns not used in any test fold: {missing_campaigns}")
        # Add missing campaigns to the fold that can best accommodate them
        best_fold_idx = find_best_fold_for_missing(folds, missing_campaigns, campaign_rows)
        folds[best_fold_idx]['test_campaigns'].extend(list(missing_campaigns))
        # Remove from train campaigns
        for missing_camp in missing_campaigns:
            if missing_camp in folds[best_fold_idx]['train_campaigns']:
                folds[best_fold_idx]['train_campaigns'].remove(missing_camp)
    
    return folds

def find_best_test_combination(all_campaigns, campaign_rows, target_test_rows, used_campaigns):
    """Find the best combination of campaigns for test set to achieve target ratio"""
    available_campaigns = [c for c in all_campaigns if c not in used_campaigns]
    
    # If no unused campaigns, allow reuse but prefer unused ones
    if not available_campaigns:
        available_campaigns = all_campaigns
        print(f"    No unused campaigns - allowing reuse from: {available_campaigns}")
    else:
        print(f"    Available unused campaigns: {available_campaigns}")
    
    best_combination = []
    best_score = float('inf')
    
    # Try different combinations, starting with single campaigns, then pairs, etc.
    from itertools import combinations
    
    max_campaigns_to_try = min(len(available_campaigns), 4)  # Limit combinations for performance
    
    for r in range(1, max_campaigns_to_try + 1):
        for combo in combinations(available_campaigns, r):
            combo_rows = sum(campaign_rows[c] for c in combo)
            # Score based on how close we are to target (prefer slightly under to over)
            score = abs(combo_rows - target_test_rows)
            if combo_rows > target_test_rows:
                score *= 1.2  # Penalize going over target
            
            if score < best_score:
                best_score = score
                best_combination = list(combo)
        
        # If we found a decent combination, don't try larger combinations
        if best_combination and best_score < target_test_rows * 0.3:  # Within 30% of target
            break
    
    # If no good combination found, just take the campaign closest to target
    if not best_combination:
        best_combination = [min(available_campaigns, key=lambda x: abs(campaign_rows[x] - target_test_rows))]
    
    return best_combination

def find_best_fold_for_missing(folds, missing_campaigns, campaign_rows):
    """Find the fold that can best accommodate missing campaigns"""
    missing_rows = sum(campaign_rows[c] for c in missing_campaigns)
    
    best_fold_idx = 0
    best_score = float('inf')
    
    for i, fold in enumerate(folds):
        current_test_rows = sum(campaign_rows[c] for c in fold['test_campaigns'])
        current_train_rows = sum(campaign_rows[c] for c in fold['train_campaigns'])
        current_total = current_test_rows + current_train_rows
        
        # Calculate new ratio if we add missing campaigns to test
        new_test_rows = current_test_rows + missing_rows
        new_total = current_total
        new_test_ratio = new_test_rows / new_total if new_total > 0 else 0
        
        # Score based on how far from 20% we'd be
        score = abs(new_test_ratio - 0.2)
        
        if score < best_score:
            best_score = score
            best_fold_idx = i
    
    return best_fold_idx

# Create outer folds
outer_folds = create_outer_folds_with_constraint(remaining_campaigns, n_folds=5)

print("Outer fold configuration:")
for fold in outer_folds:
    print(f"  Fold {fold['fold']}: Train={len(fold['train_campaigns'])}, Test={len(fold['test_campaigns'])}")
    print(f"    Train campaigns: {fold['train_campaigns']}")
    print(f"    Test campaigns: {fold['test_campaigns']}")

# ============================================================================
# STEP 4: Create Inner Folds (3-Fold) for Each Outer Fold
# ============================================================================
print("\n" + "="*60)
print("STEP 4: CREATING INNER FOLDS (3-FOLD PER OUTER FOLD)")
print("="*60)

def create_inner_folds_random(campaigns, n_folds=3):
    """Create inner folds with balanced splits aiming for 80:20 train:test ratio"""
    
    # Get row counts for each campaign (filtered data)
    campaign_rows = {}
    for campaign in campaigns:
        campaign_data = df[df['Campaign'] == campaign]
        campaign_rows[campaign] = filter_invalid_data(campaign_data).shape[0]
    
    total_rows = sum(campaign_rows.values())
    target_test_rows_per_fold = total_rows * 0.2 / n_folds  # 20% of total divided by number of folds
    
    print(f"      Inner fold total rows: {total_rows:,}, target test per fold: {target_test_rows_per_fold:,.0f}")
    
    # Sort campaigns by row count for better distribution
    sorted_campaigns = sorted(campaigns, key=lambda x: campaign_rows[x], reverse=True)
    
    folds = []
    campaigns_used_in_test = set()
    
    for fold_idx in range(n_folds):
        if fold_idx == n_folds - 1:
            # Last fold gets remaining unused campaigns
            unused_campaigns = set(campaigns) - campaigns_used_in_test
            if unused_campaigns:
                test_campaigns = list(unused_campaigns)
            else:
                # Find best combination from all campaigns
                test_campaigns = find_best_inner_test_combination(campaigns, campaign_rows, target_test_rows_per_fold, set())
        else:
            # Find best combination for this fold
            test_campaigns = find_best_inner_test_combination(campaigns, campaign_rows, target_test_rows_per_fold, campaigns_used_in_test)
        
        train_campaigns = [c for c in campaigns if c not in test_campaigns]
        
        # Calculate actual ratios
        test_rows = sum(campaign_rows[c] for c in test_campaigns)
        train_rows = sum(campaign_rows[c] for c in train_campaigns)
        total_fold_rows = train_rows + test_rows
        
        test_pct = (test_rows / total_fold_rows * 100) if total_fold_rows > 0 else 0
        train_pct = (train_rows / total_fold_rows * 100) if total_fold_rows > 0 else 0
        
        print(f"        Inner fold {fold_idx + 1}: Test={test_campaigns} ({test_pct:.1f}%), Train={train_campaigns} ({train_pct:.1f}%)")
        
        folds.append({
            'fold': fold_idx + 1,
            'train_campaigns': train_campaigns,
            'test_campaigns': test_campaigns
        })
        
        campaigns_used_in_test.update(test_campaigns)
    
    return folds

def find_best_inner_test_combination(all_campaigns, campaign_rows, target_test_rows, used_campaigns):
    """Find the best combination of campaigns for inner test set"""
    available_campaigns = [c for c in all_campaigns if c not in used_campaigns]
    
    # If no unused campaigns, allow reuse
    if not available_campaigns:
        available_campaigns = all_campaigns
    
    best_combination = []
    best_score = float('inf')
    
    # Try different combinations
    from itertools import combinations
    
    max_campaigns_to_try = min(len(available_campaigns), 3)  # Limit for inner folds
    
    for r in range(1, max_campaigns_to_try + 1):
        for combo in combinations(available_campaigns, r):
            combo_rows = sum(campaign_rows[c] for c in combo)
            score = abs(combo_rows - target_test_rows)
            if combo_rows > target_test_rows:
                score *= 1.1  # Slight penalty for going over
            
            if score < best_score:
                best_score = score
                best_combination = list(combo)
        
        # Stop if we found a good combination
        if best_combination and best_score < target_test_rows * 0.4:
            break
    
    # Fallback to single campaign closest to target
    if not best_combination:
        best_combination = [min(available_campaigns, key=lambda x: abs(campaign_rows[x] - target_test_rows))]
    
    return best_combination

# Create and save all folds
print("Processing outer folds and creating inner folds...")

# Store inner folds for diagnostics
all_inner_folds = {}

for outer_fold in outer_folds:
    fold_num = outer_fold['fold']
    print(f"\n--- Processing Outer Fold {fold_num} ---")
    
    # Create outer fold directory
    fold_dir = os.path.join(FINAL_OUT_PATH, f"kfold{fold_num}")
    os.makedirs(fold_dir, exist_ok=True)
    
    # Get outer fold data
    outer_train_campaigns = outer_fold['train_campaigns']
    outer_test_campaigns = outer_fold['test_campaigns']
    
    print(f"Outer train campaigns: {outer_train_campaigns}")
    print(f"Outer test campaigns: {outer_test_campaigns}")
    
    # Create outer fold datasets
    outer_train_df = df[df['Campaign'].isin(outer_train_campaigns)].copy()
    outer_test_df = df[df['Campaign'].isin(outer_test_campaigns)].copy()
    
    print(f"Outer train data (before filtering): {outer_train_df.shape}")
    print(f"Outer test data (before filtering): {outer_test_df.shape}")
    
    # Filter outer fold data
    outer_train_df_filtered = filter_invalid_data(outer_train_df)
    outer_test_df_filtered = filter_invalid_data(outer_test_df)
    
    print(f"Outer train data (after filtering): {outer_train_df_filtered.shape}")
    print(f"Outer test data (after filtering): {outer_test_df_filtered.shape}")
    
    # Save outer fold data
    outer_train_path = os.path.join(fold_dir, "train.csv")
    outer_test_path = os.path.join(fold_dir, "test.csv")
    
    outer_train_df_filtered.to_csv(outer_train_path, index=False)
    outer_test_df_filtered.to_csv(outer_test_path, index=False)
    
    print(f"Saved outer fold train.csv to: {outer_train_path}")
    print(f"Saved outer fold test.csv to: {outer_test_path}")
    
    # Create inner folds from outer training campaigns
    if len(outer_train_campaigns) >= 3:
        inner_folds = create_inner_folds_random(outer_train_campaigns, n_folds=3)
        all_inner_folds[fold_num] = inner_folds  # Store for diagnostics
        
        print(f"  Creating {len(inner_folds)} inner folds:")
        for inner_fold in inner_folds:
            inner_fold_num = inner_fold['fold']
            inner_train_campaigns = inner_fold['train_campaigns']
            inner_test_campaigns = inner_fold['test_campaigns']
            
            print(f"    Inner fold {inner_fold_num}: Train={len(inner_train_campaigns)}, Test={len(inner_test_campaigns)}")
            print(f"      Train campaigns: {inner_train_campaigns}")
            print(f"      Test campaigns: {inner_test_campaigns}")
            
            # Create inner fold directory
            inner_fold_dir = os.path.join(fold_dir, f"innerfold{inner_fold_num}")
            os.makedirs(inner_fold_dir, exist_ok=True)
            
            # Create inner fold datasets
            inner_train_df = df[df['Campaign'].isin(inner_train_campaigns)].copy()
            inner_test_df = df[df['Campaign'].isin(inner_test_campaigns)].copy()
            
            # Filter inner fold data
            inner_train_df_filtered = filter_invalid_data(inner_train_df)
            inner_test_df_filtered = filter_invalid_data(inner_test_df)
            
            # Save inner fold data
            inner_train_path = os.path.join(inner_fold_dir, "train.csv")
            inner_test_path = os.path.join(inner_fold_dir, "test.csv")
            
            inner_train_df_filtered.to_csv(inner_train_path, index=False)
            inner_test_df_filtered.to_csv(inner_test_path, index=False)
            
            print(f"      Saved inner train.csv to: {inner_train_path}")
            print(f"      Saved inner test.csv to: {inner_test_path}")
    else:
        print(f"  WARNING: Outer fold {fold_num} has only {len(outer_train_campaigns)} training campaigns - skipping inner folds (need at least 3)")
        all_inner_folds[fold_num] = []  # No inner folds

# ============================================================================
# STEP 5: Create Diagnostics CSV
# ============================================================================
print("\n" + "="*60)
print("STEP 5: CREATING DIAGNOSTICS CSV")
print("="*60)

# Create diagnostics data
diagnostics_data = []

# Add evaluation set info
eval_rows = evaluation_df_filtered.shape[0]
diagnostics_data.append({
    'fold_type': 'evaluation',
    'fold_number': 'N/A',
    'split_type': 'evaluation',
    'campaigns': ', '.join(evaluation_campaigns),
    'num_campaigns': len(evaluation_campaigns),
    'num_rows': eval_rows,
    'percentage_of_total': (eval_rows / df.shape[0]) * 100
})

# Calculate total rows in nested CV (excluding evaluation)
nested_cv_df = df[~df['Campaign'].isin(evaluation_campaigns)]
nested_cv_rows_total = filter_invalid_data(nested_cv_df).shape[0]

# Add outer fold info
for outer_fold in outer_folds:
    fold_num = outer_fold['fold']
    
    # Get row counts for this outer fold
    outer_train_df_temp = df[df['Campaign'].isin(outer_fold['train_campaigns'])]
    outer_test_df_temp = df[df['Campaign'].isin(outer_fold['test_campaigns'])]
    
    outer_train_rows = filter_invalid_data(outer_train_df_temp).shape[0]
    outer_test_rows = filter_invalid_data(outer_test_df_temp).shape[0]
    outer_total_rows = outer_train_rows + outer_test_rows
    
    # Outer train info
    diagnostics_data.append({
        'fold_type': 'outer',
        'fold_number': fold_num,
        'split_type': 'train',
        'campaigns': ', '.join(outer_fold['train_campaigns']),
        'num_campaigns': len(outer_fold['train_campaigns']),
        'num_rows': outer_train_rows,
        'percentage_of_total': (outer_train_rows / nested_cv_rows_total) * 100,
        'percentage_of_fold': (outer_train_rows / outer_total_rows) * 100 if outer_total_rows > 0 else 0
    })
    
    # Outer test info
    diagnostics_data.append({
        'fold_type': 'outer',
        'fold_number': fold_num,
        'split_type': 'test',
        'campaigns': ', '.join(outer_fold['test_campaigns']),
        'num_campaigns': len(outer_fold['test_campaigns']),
        'num_rows': outer_test_rows,
        'percentage_of_total': (outer_test_rows / nested_cv_rows_total) * 100,
        'percentage_of_fold': (outer_test_rows / outer_total_rows) * 100 if outer_total_rows > 0 else 0
    })
    
    # Add inner fold info (if inner folds were created)
    if len(outer_fold['train_campaigns']) >= 3 and fold_num in all_inner_folds:
        inner_folds = all_inner_folds[fold_num]  # Use stored inner folds
        
        # Calculate total rows for inner folds (same as outer train)
        inner_total_rows = outer_train_rows
        
        for inner_fold in inner_folds:
            inner_fold_num = inner_fold['fold']
            
            # Get row counts for this inner fold
            inner_train_df_temp = df[df['Campaign'].isin(inner_fold['train_campaigns'])]
            inner_test_df_temp = df[df['Campaign'].isin(inner_fold['test_campaigns'])]
            
            inner_train_rows = filter_invalid_data(inner_train_df_temp).shape[0]
            inner_test_rows = filter_invalid_data(inner_test_df_temp).shape[0]
            
            # Inner train info
            diagnostics_data.append({
                'fold_type': 'inner',
                'fold_number': f"{fold_num}.{inner_fold_num}",
                'split_type': 'train',
                'campaigns': ', '.join(inner_fold['train_campaigns']),
                'num_campaigns': len(inner_fold['train_campaigns']),
                'num_rows': inner_train_rows,
                'percentage_of_total': (inner_train_rows / nested_cv_rows_total) * 100,
                'percentage_of_fold': (inner_train_rows / inner_total_rows) * 100 if inner_total_rows > 0 else 0
            })
            
            # Inner test info
            diagnostics_data.append({
                'fold_type': 'inner',
                'fold_number': f"{fold_num}.{inner_fold_num}",
                'split_type': 'test',
                'campaigns': ', '.join(inner_fold['test_campaigns']),
                'num_campaigns': len(inner_fold['test_campaigns']),
                'num_rows': inner_test_rows,
                'percentage_of_total': (inner_test_rows / nested_cv_rows_total) * 100,
                'percentage_of_fold': (inner_test_rows / inner_total_rows) * 100 if inner_total_rows > 0 else 0
            })

# Create DataFrame and save diagnostics
import pandas as pd
diagnostics_df = pd.DataFrame(diagnostics_data)

# Round percentages for readability
diagnostics_df['percentage_of_total'] = diagnostics_df['percentage_of_total'].round(2)
if 'percentage_of_fold' in diagnostics_df.columns:
    diagnostics_df['percentage_of_fold'] = diagnostics_df['percentage_of_fold'].round(2)

# Save diagnostics CSV
diagnostics_path = os.path.join(FINAL_OUT_PATH, "kfold_diagnostics.csv")
diagnostics_df.to_csv(diagnostics_path, index=False)

print(f"Saved diagnostics to: {diagnostics_path}")
print("\nDiagnostics preview:")
print(diagnostics_df.head(10).to_string(index=False))

# ============================================================================
print("\n" + "="*60)
print("NESTED CROSS-VALIDATION SETUP COMPLETED")
print("="*60)

print(f"Directory structure created in: {FINAL_OUT_PATH}")
print(f"")
print(f"Structure:")
print(f"├── evaluation/")
print(f"│   └── evaluation.csv ({len(evaluation_campaigns)} campaigns: {evaluation_campaigns})")

for fold in outer_folds:
    fold_num = fold['fold']
    train_count = len(fold['train_campaigns'])
    test_count = len(fold['test_campaigns'])
    print(f"├── kfold{fold_num}/")
    print(f"│   ├── train.csv ({train_count} campaigns)")
    print(f"│   ├── test.csv ({test_count} campaigns)")
    if train_count >= 3:
        print(f"│   ├── innerfold1/ (train.csv, test.csv)")
        print(f"│   ├── innerfold2/ (train.csv, test.csv)")
        print(f"│   └── innerfold3/ (train.csv, test.csv)")
    else:
        print(f"│   └── (no inner folds - insufficient campaigns)")

# Verification
print(f"\nVerification:")
all_test_campaigns = set()
for fold in outer_folds:
    all_test_campaigns.update(fold['test_campaigns'])

missing_from_test = set(remaining_campaigns) - all_test_campaigns
if missing_from_test:
    print(f"❌ ERROR: Campaigns never used in test: {missing_from_test}")
else:
    print(f"✅ All {len(remaining_campaigns)} campaigns appear in at least one outer test fold")

print(f"✅ Hold-out evaluation set: {len(evaluation_campaigns)} campaigns")
print(f"✅ Nested CV structure: 5 outer folds × 3 inner folds each")
print(f"✅ Diagnostics CSV created: kfold_diagnostics.csv")
print(f"✅ All data filtered and saved successfully")

Starting Nested Cross-Validation dataset splitting...
Original dataset shape: (193027, 59)
All campaigns in dataset: ['ACMEV', 'ASIA-AQ', 'BBOP', 'CACTI', 'CARES', 'DC3', 'DISCOVERAQ-California', 'DISCOVERAQ-Texas', 'FIREXAQ', 'NAAMES(2015)', 'NAAMES(2016)', 'NAAMES(2017)', 'TCAP2013']

STEP 1: CREATING HOLD-OUT EVALUATION SET
Evaluation campaigns (hold-out): ['BBOP', 'CACTI', 'CARES', 'TCAP2013', 'ACMEV']
Evaluation data shape (before filtering): (12562, 59)
Evaluation data shape (after filtering): (12562, 59)
Evaluation rows removed: 0
Saved evaluation.csv to: C:\Users\haika\Desktop\May_Research\MAC Machine Learning Model Code\split_datasets\data_split_strat1\MAC_DATASET_LLOD_FILTERED\kfold_based_split\evaluation\evaluation.csv

STEP 2: PREPARING CAMPAIGNS FOR NESTED CV
Remaining campaigns for nested CV: ['ASIA-AQ', 'DC3', 'DISCOVERAQ-California', 'DISCOVERAQ-Texas', 'FIREXAQ', 'NAAMES(2015)', 'NAAMES(2016)', 'NAAMES(2017)']
Number of campaigns for nested CV: 8

STEP 3: CREATING OUTE

KeyboardInterrupt: 

SPLITTING METHOD 2: CAMPAIGN_BASED Split

In [ ]:
FINAL_OUT_PATH = os.path.join(output_path, "campaign_based_split")
os.makedirs(FINAL_OUT_PATH, exist_ok=True)

print("Starting dataset splitting...")
print(f"Original dataset shape: {df.shape}")

# ============================================================================
# METHOD 2: Split by Entire Campaigns (3-way split)
# ============================================================================
print("\n" + "="*60)
print("METHOD 2: Splitting by Entire Campaigns (Train/Validation/Evaluation)")
print("="*60)

# Get train, validation, and evaluation campaign lists from the CSV
train_campaigns_str = df_campaign_based_info['train_campaigns'].iloc[0]
validation_campaigns_str = df_campaign_based_info['validation_campaigns'].iloc[0]
evaluation_campaigns_str = df_campaign_based_info['evaluation_campaigns'].iloc[0]

# Convert comma-separated strings to lists
train_campaigns = [camp.strip() for camp in train_campaigns_str.split(',')]
validation_campaigns = [camp.strip() for camp in validation_campaigns_str.split(',')]
evaluation_campaigns = [camp.strip() for camp in evaluation_campaigns_str.split(',')]

print(f"Train campaigns: {train_campaigns}")
print(f"Validation campaigns: {validation_campaigns}")
print(f"Evaluation campaigns: {evaluation_campaigns}")

# Split the dataset by campaigns
train_df = df[df['Campaign'].isin(train_campaigns)].copy()
validation_df = df[df['Campaign'].isin(validation_campaigns)].copy()
evaluation_df = df[df['Campaign'].isin(evaluation_campaigns)].copy()

print(f"Train set shape: {train_df.shape}")
print(f"Validation set shape: {validation_df.shape}")
print(f"Evaluation set shape: {evaluation_df.shape}")

# Verify all campaigns are accounted for
all_campaigns_in_splits = set(train_campaigns + validation_campaigns + evaluation_campaigns)
all_campaigns_in_data = set(df['Campaign'].unique())
missing_campaigns = all_campaigns_in_data - all_campaigns_in_splits
extra_campaigns = all_campaigns_in_splits - all_campaigns_in_data

if missing_campaigns:
    print(f"WARNING: Missing campaigns in split files: {missing_campaigns}")
if extra_campaigns:
    print(f"WARNING: Extra campaigns in split files: {extra_campaigns}")

# Check for campaign overlaps
train_set = set(train_campaigns)
validation_set = set(validation_campaigns)
evaluation_set = set(evaluation_campaigns)

train_val_overlap = train_set & validation_set
train_eval_overlap = train_set & evaluation_set
val_eval_overlap = validation_set & evaluation_set

if train_val_overlap:
    print(f"WARNING: Overlap between train and validation: {train_val_overlap}")
if train_eval_overlap:
    print(f"WARNING: Overlap between train and evaluation: {train_eval_overlap}")
if val_eval_overlap:
    print(f"WARNING: Overlap between validation and evaluation: {val_eval_overlap}")

print(f"\nFiltering data...")
print(f"Before filtering - Train: {train_df.shape[0]:,} rows, Validation: {validation_df.shape[0]:,} rows, Evaluation: {evaluation_df.shape[0]:,} rows")

# Filter out rows with NaN, empty values, -2222, or -9999 in model_columns
def filter_invalid_data(df):
    # Create mask for valid data
    mask = True
    for col in model_columns:
        if col in df.columns:
            # Check for NaN, -2222, -9999, and empty values
            col_mask = (~df[col].isna()) & (df[col] != -2222) & (df[col] != -9999) & (df[col] != '')
            mask = mask & col_mask
        else:
            print(f"Warning: Column '{col}' not found in dataset")
    return df[mask].copy()

train_df_filtered = filter_invalid_data(train_df)
validation_df_filtered = filter_invalid_data(validation_df)
evaluation_df_filtered = filter_invalid_data(evaluation_df)

print(f"After filtering - Train: {train_df_filtered.shape[0]:,} rows, Validation: {validation_df_filtered.shape[0]:,} rows, Evaluation: {evaluation_df_filtered.shape[0]:,} rows")
print(f"Rows removed - Train: {train_df.shape[0] - train_df_filtered.shape[0]:,}, Validation: {validation_df.shape[0] - validation_df_filtered.shape[0]:,}, Evaluation: {evaluation_df.shape[0] - evaluation_df_filtered.shape[0]:,}")

# Save results
train_path = os.path.join(FINAL_OUT_PATH, "train.csv")
validation_path = os.path.join(FINAL_OUT_PATH, "validation.csv")
evaluation_path = os.path.join(FINAL_OUT_PATH, "evaluation.csv")

train_df_filtered.to_csv(train_path, index=False)
validation_df_filtered.to_csv(validation_path, index=False)
evaluation_df_filtered.to_csv(evaluation_path, index=False)

print(f"\nSaved train.csv to: {train_path}")
print(f"Saved validation.csv to: {validation_path}")
print(f"Saved evaluation.csv to: {evaluation_path}")

# Print final summary
print(f"\n" + "="*60)
print("CAMPAIGN-BASED SPLIT SUMMARY")
print("="*60)
total_filtered_rows = train_df_filtered.shape[0] + validation_df_filtered.shape[0] + evaluation_df_filtered.shape[0]
print(f"Total rows after filtering: {total_filtered_rows:,}")
print(f"Train: {train_df_filtered.shape[0]:,} rows ({100*train_df_filtered.shape[0]/total_filtered_rows:.1f}%)")
print(f"Validation: {validation_df_filtered.shape[0]:,} rows ({100*validation_df_filtered.shape[0]/total_filtered_rows:.1f}%)")
print(f"Evaluation: {evaluation_df_filtered.shape[0]:,} rows ({100*evaluation_df_filtered.shape[0]/total_filtered_rows:.1f}%)")

Starting dataset splitting...
Original dataset shape: (193027, 59)

METHOD 2: Splitting by Entire Campaigns (Train/Validation/Evaluation)
Train campaigns: ['ASIA-AQ', 'DC3', 'DISCOVERAQ-California', 'DISCOVERAQ-Texas']
Validation campaigns: ['FIREXAQ', 'NAAMES(2015)', 'NAAMES(2016)', 'NAAMES(2017)']
Evaluation campaigns: ['TCAP2013', 'CACTI', 'BBOP', 'ACMEV', 'CARES']
Train set shape: (143023, 59)
Validation set shape: (37442, 59)
Evaluation set shape: (12562, 59)

Filtering data...
Before filtering - Train: 143,023 rows, Validation: 37,442 rows, Evaluation: 12,562 rows
After filtering - Train: 143,023 rows, Validation: 37,442 rows, Evaluation: 12,562 rows
Rows removed - Train: 0, Validation: 0, Evaluation: 0

Saved train.csv to: C:\Users\haika\Desktop\May_Research\MAC Machine Learning Model Code\split_datasets\data_split_start1\MAC_DATASET_LLOD_FILTERED\campaign_based_split\train.csv
Saved validation.csv to: C:\Users\haika\Desktop\May_Research\MAC Machine Learning Model Code\split_dat

SPLITTING METHOD 3: DAY_BASED Split

In [ ]:
FINAL_OUT_PATH = os.path.join(output_path, "day_based_split")
os.makedirs(FINAL_OUT_PATH, exist_ok=True)

# ============================================================================
# METHOD 3: Split by Dates with Hold-out Campaigns (Train/Test/Evaluation)
# ============================================================================
print("\n" + "="*60)
print("METHOD 3: Splitting by Dates with Hold-out Campaigns (Train/Test/Evaluation)")
print("="*60)

print("Reading day-based split configuration...")

# Read the day-based info list
print(f"Day-based info shape: {df_day_based_info.shape}")
print(f"Columns: {df_day_based_info.columns.tolist()}")

# Debug: Show first few rows
print(f"\nFirst 5 rows of day_based_info:")
print(df_day_based_info.head())

# Check if hold_out column exists, if not create it with default value 0
if 'hold_out' not in df_day_based_info.columns:
    print("WARNING: 'hold_out' column not found. Creating with default value 0 (no hold-out campaigns)")
    df_day_based_info['hold_out'] = 0

# Separate hold-out campaigns from date-split campaigns
holdout_campaigns = df_day_based_info[df_day_based_info['hold_out'] == 1]['campaign'].tolist()
date_split_campaigns = df_day_based_info[df_day_based_info['hold_out'] == 0]

print(f"\nHold-out campaigns (entire campaigns go to evaluation): {holdout_campaigns}")
print(f"Date-split campaigns (split by specific dates): {date_split_campaigns['campaign'].tolist()}")

# Initialize datasets
train_df_list = []
test_df_list = []
evaluation_df_list = []

# Process hold-out campaigns first
if holdout_campaigns:
    print(f"\nProcessing hold-out campaigns...")
    for campaign in holdout_campaigns:
        campaign_data = df[df['Campaign'] == campaign].copy()
        if campaign_data.shape[0] > 0:
            evaluation_df_list.append(campaign_data)
            print(f"  {campaign}: {campaign_data.shape[0]:,} rows → evaluation.csv")
        else:
            print(f"  WARNING: No data found for hold-out campaign '{campaign}'")

# Helper function to safely parse date strings
def parse_dates_safely(date_str):
    """Safely parse date string, handling NaN and various formats"""
    if pd.isna(date_str):
        return []
    
    # Convert to string if it's not already
    if not isinstance(date_str, str):
        date_str = str(date_str)
    
    # Check for common NaN representations
    if date_str.strip().lower() in ['nan', 'none', '']:
        return []
    
    # Split by comma and clean up
    dates = [date.strip() for date in date_str.split(',') if date.strip()]
    return dates

# Process date-split campaigns
print(f"\nProcessing date-split campaigns...")
for _, row in date_split_campaigns.iterrows():
    campaign = row['campaign']
    train_dates_str = row['train_dates']
    test_dates_str = row['test_dates']
    
    print(f"\n--- Processing Campaign: {campaign} ---")
    
    # Get campaign data
    campaign_data = df[df['Campaign'] == campaign].copy()
    if campaign_data.shape[0] == 0:
        print(f"  WARNING: No data found for campaign '{campaign}'")
        continue
    
    print(f"  Campaign data shape: {campaign_data.shape}")
    
    # Parse train and test dates safely
    train_dates = parse_dates_safely(train_dates_str)
    test_dates = parse_dates_safely(test_dates_str)
    
    print(f"  Train dates: {train_dates}")
    print(f"  Test dates: {test_dates}")
    
    # Convert Date column to string for consistent comparison
    campaign_data['Date'] = campaign_data['Date'].astype(str)
    
    # Get train data for this campaign
    if train_dates:
        campaign_train_data = campaign_data[campaign_data['Date'].isin(train_dates)].copy()
        if campaign_train_data.shape[0] > 0:
            train_df_list.append(campaign_train_data)
            print(f"  Train data: {campaign_train_data.shape[0]:,} rows")
        else:
            print(f"  WARNING: No train data found for dates {train_dates}")
    else:
        print(f"  No train dates specified")
    
    # Get test data for this campaign
    if test_dates:
        campaign_test_data = campaign_data[campaign_data['Date'].isin(test_dates)].copy()
        if campaign_test_data.shape[0] > 0:
            test_df_list.append(campaign_test_data)
            print(f"  Test data: {campaign_test_data.shape[0]:,} rows")
        else:
            print(f"  WARNING: No test data found for dates {test_dates}")
    else:
        print(f"  No test dates specified")
    
    # Verify no date overlap for this campaign
    train_set = set(train_dates)
    test_set = set(test_dates)
    date_overlap = train_set & test_set
    if date_overlap:
        print(f"  ERROR: Date overlap in campaign {campaign}: {date_overlap}")
    
    # Check if all campaign dates are accounted for
    all_campaign_dates = set(campaign_data['Date'].unique())
    assigned_dates = train_set | test_set
    unassigned_dates = all_campaign_dates - assigned_dates
    if unassigned_dates:
        print(f"  INFO: Unassigned dates in {campaign}: {sorted(unassigned_dates)} ({len(unassigned_dates)} dates)")

# Combine all dataframes
print(f"\n" + "="*40)
print("COMBINING DATASETS")
print("="*40)

if train_df_list:
    train_df = pd.concat(train_df_list, ignore_index=True)
    print(f"Combined train data: {train_df.shape[0]:,} rows from {len(train_df_list)} campaign splits")
else:
    train_df = pd.DataFrame()
    print("No train data found")

if test_df_list:
    test_df = pd.concat(test_df_list, ignore_index=True)
    print(f"Combined test data: {test_df.shape[0]:,} rows from {len(test_df_list)} campaign splits")
else:
    test_df = pd.DataFrame()
    print("No test data found")

if evaluation_df_list:
    evaluation_df = pd.concat(evaluation_df_list, ignore_index=True)
    print(f"Combined evaluation data: {evaluation_df.shape[0]:,} rows from {len(evaluation_df_list)} hold-out campaigns")
else:
    evaluation_df = pd.DataFrame()
    print("No evaluation data found")

# Check for overlaps between final datasets
print(f"\n" + "="*40)
print("OVERLAP VERIFICATION")
print("="*40)

def check_campaign_overlap(df1, df2, name1, name2):
    if df1.shape[0] > 0 and df2.shape[0] > 0:
        campaigns1 = set(df1['Campaign'].unique())
        campaigns2 = set(df2['Campaign'].unique())
        overlap = campaigns1 & campaigns2
        if overlap:
            print(f"ERROR: Campaign overlap between {name1} and {name2}: {overlap}")
        else:
            print(f"✓ No campaign overlap between {name1} and {name2}")

check_campaign_overlap(train_df, test_df, "train", "test")
check_campaign_overlap(train_df, evaluation_df, "train", "evaluation")
check_campaign_overlap(test_df, evaluation_df, "test", "evaluation")

# Define model columns for filtering
model_columns = ['N1', 'N2', 'N3', 'F_N1', 'F_N2', 'F_N3', 'V1', 'V2', 'V3', 'F_V1', 'F_V2', 'F_V3', 'b_scat_red', 'b_scat_blue', 'b_scat_green', 'b_abs_red', 'b_abs_blue', 'b_abs_green', 'MAC_bc']

# Filter invalid data
def filter_invalid_data(df):
    """Filter out rows with NaN, empty values, -2222, or -9999 in model_columns"""
    if df.shape[0] == 0:
        return df
    
    mask = True
    for col in model_columns:
        if col in df.columns:
            col_mask = (~df[col].isna()) & (df[col] != -2222) & (df[col] != -9999) & (df[col] != '')
            mask = mask & col_mask
        else:
            print(f"Warning: Column '{col}' not found in dataset")
    return df[mask].copy()

print(f"\n" + "="*40)
print("FILTERING DATA")
print("="*40)

print(f"Before filtering:")
print(f"  Train: {train_df.shape[0]:,} rows")
print(f"  Test: {test_df.shape[0]:,} rows") 
print(f"  Evaluation: {evaluation_df.shape[0]:,} rows")

train_df_filtered = filter_invalid_data(train_df)
test_df_filtered = filter_invalid_data(test_df)
evaluation_df_filtered = filter_invalid_data(evaluation_df)

print(f"\nAfter filtering:")
print(f"  Train: {train_df_filtered.shape[0]:,} rows")
print(f"  Test: {test_df_filtered.shape[0]:,} rows")
print(f"  Evaluation: {evaluation_df_filtered.shape[0]:,} rows")

print(f"\nRows removed:")
print(f"  Train: {train_df.shape[0] - train_df_filtered.shape[0]:,}")
print(f"  Test: {test_df.shape[0] - test_df_filtered.shape[0]:,}")
print(f"  Evaluation: {evaluation_df.shape[0] - evaluation_df_filtered.shape[0]:,}")

# Calculate final percentages
total_filtered_rows = train_df_filtered.shape[0] + test_df_filtered.shape[0] + evaluation_df_filtered.shape[0]

if total_filtered_rows > 0:
    train_pct = (train_df_filtered.shape[0] / total_filtered_rows) * 100
    test_pct = (test_df_filtered.shape[0] / total_filtered_rows) * 100
    eval_pct = (evaluation_df_filtered.shape[0] / total_filtered_rows) * 100
    
    print(f"\nFinal split percentages:")
    print(f"  Train: {train_df_filtered.shape[0]:,} rows ({train_pct:.1f}%)")
    print(f"  Test: {test_df_filtered.shape[0]:,} rows ({test_pct:.1f}%)")
    print(f"  Evaluation: {evaluation_df_filtered.shape[0]:,} rows ({eval_pct:.1f}%)")
else:
    print(f"\nERROR: No rows remaining after filtering!")

# Save results
train_path = os.path.join(FINAL_OUT_PATH, "train.csv")
test_path = os.path.join(FINAL_OUT_PATH, "test.csv")
evaluation_path = os.path.join(FINAL_OUT_PATH, "evaluation.csv")

train_df_filtered.to_csv(train_path, index=False)
test_df_filtered.to_csv(test_path, index=False)
evaluation_df_filtered.to_csv(evaluation_path, index=False)

print(f"\nSaved files:")
print(f"  train.csv: {train_path}")
print(f"  test.csv: {test_path}")
print(f"  evaluation.csv: {evaluation_path}")

# Print summary by campaign
print(f"\n" + "="*60)
print("FINAL SUMMARY BY CAMPAIGN")
print("="*60)

all_campaigns = set(df['Campaign'].unique())
for campaign in sorted(all_campaigns):
    train_count = train_df_filtered[train_df_filtered['Campaign'] == campaign].shape[0] if train_df_filtered.shape[0] > 0 else 0
    test_count = test_df_filtered[test_df_filtered['Campaign'] == campaign].shape[0] if test_df_filtered.shape[0] > 0 else 0
    eval_count = evaluation_df_filtered[evaluation_df_filtered['Campaign'] == campaign].shape[0] if evaluation_df_filtered.shape[0] > 0 else 0
    total_count = train_count + test_count + eval_count
    
    if total_count > 0:
        status = "HOLD-OUT" if campaign in holdout_campaigns else "DATE-SPLIT"
        print(f"{campaign:20s} ({status:10s}): Train={train_count:6,}, Test={test_count:6,}, Eval={eval_count:6,}, Total={total_count:6,}")

print(f"\n" + "="*60)
print("DATE-BASED SPLIT WITH HOLD-OUT COMPLETED")
print("="*60)


METHOD 3: Splitting by Dates with Hold-out Campaigns (Train/Test/Evaluation)
Reading day-based split configuration...
Day-based info shape: (13, 4)
Columns: ['campaign', 'train_dates', 'test_dates', 'hold_out']

First 5 rows of day_based_info:
   campaign                                        train_dates  \
0      BBOP  20130715, 20130719, 20130723, 20130725, 201307...   
1     ACMEV  20150802, 20150806, 20150807, 20150827, 201509...   
2     CACTI                                           20181114   
3  TCAP2013  20130207, 20130213, 20130215, 20130218, 201302...   
4     CARES  20100610, 20100612, 20100614, 20100615, 201006...   

           test_dates  hold_out  
0            20130717         1  
1  20150808, 20150825         1  
2                 NaN         1  
3  20130219, 20130225         1  
4  20100608, 20100618         1  

Hold-out campaigns (entire campaigns go to evaluation): ['BBOP', 'ACMEV', 'CACTI', 'TCAP2013', 'CARES']
Date-split campaigns (split by specific dates): [

SPLITTING METHOD 4: RANDOM_BASED Split

In [ ]:
# SPLITTING METHOD 4: RANDOM_BASED Split
FINAL_OUT_PATH = os.path.join(output_path, "random_based_split")
os.makedirs(FINAL_OUT_PATH, exist_ok=True)

print("\n" + "="*60)
print("METHOD 4: Random-Based Split (80:20 Train/Test + Hold-out Evaluation)")
print("="*60)

# Read held-out campaigns from random_based_info CSV
print("Reading random-based split configuration...")
print(f"Random-based info shape: {df_random_based_info.shape}")
print(f"Columns: {df_random_based_info.columns.tolist()}")

# Get held-out campaigns
held_out_str = df_random_based_info['held_out'].iloc[0]
held_out_campaigns = [camp.strip() for camp in held_out_str.split(',') if camp.strip()]

print(f"Held-out campaigns (go to evaluation): {held_out_campaigns}")

# Split data into held-out (evaluation) and remaining (for random split)
evaluation_df = df[df['Campaign'].isin(held_out_campaigns)].copy()
remaining_df = df[~df['Campaign'].isin(held_out_campaigns)].copy()

print(f"Evaluation data (held-out campaigns): {evaluation_df.shape[0]:,} rows")
print(f"Remaining data (for random split): {remaining_df.shape[0]:,} rows")

# Verify all campaigns are accounted for
all_campaigns_in_data = set(df['Campaign'].unique())
held_out_set = set(held_out_campaigns)
remaining_campaigns = all_campaigns_in_data - held_out_set

print(f"Remaining campaigns for random split: {sorted(remaining_campaigns)}")

# Check for missing campaigns
missing_campaigns = held_out_set - all_campaigns_in_data
if missing_campaigns:
    print(f"WARNING: Held-out campaigns not found in data: {missing_campaigns}")

# Random 80:20 split of remaining data
from sklearn.model_selection import train_test_split

print(f"\nPerforming random 80:20 split on remaining {remaining_df.shape[0]:,} rows...")

if remaining_df.shape[0] > 0:
    train_df, test_df = train_test_split(
        remaining_df, 
        test_size=0.2, 
        random_state=42,  # For reproducibility
        shuffle=True
    )
    
    print(f"Random split results:")
    print(f"  Train: {train_df.shape[0]:,} rows ({100*train_df.shape[0]/remaining_df.shape[0]:.1f}%)")
    print(f"  Test: {test_df.shape[0]:,} rows ({100*test_df.shape[0]/remaining_df.shape[0]:.1f}%)")
else:
    print("ERROR: No remaining data for random split!")
    train_df = pd.DataFrame()
    test_df = pd.DataFrame()

# Verify campaign distribution in random splits
print(f"\nCampaign distribution in random splits:")
if train_df.shape[0] > 0:
    train_campaigns = train_df['Campaign'].value_counts()
    print(f"Train campaigns:")
    for campaign, count in train_campaigns.items():
        pct = (count / train_df.shape[0]) * 100
        print(f"  {campaign}: {count:,} rows ({pct:.1f}%)")

if test_df.shape[0] > 0:
    test_campaigns = test_df['Campaign'].value_counts()
    print(f"Test campaigns:")
    for campaign, count in test_campaigns.items():
        pct = (count / test_df.shape[0]) * 100
        print(f"  {campaign}: {count:,} rows ({pct:.1f}%)")

# Define model columns for filtering
model_columns = ['N1', 'N2', 'N3', 'N_total', 'V1', 'V2', 'V3', 'V_total', 'SAE', 'AAE', 
                'SSA_red', 'SSA_blue', 'SSA_green', 'b_scat_red', 'b_scat_blue', 'b_scat_green', 
                'b_abs_red', 'b_abs_blue', 'b_abs_green', 'MAC_bc']

# Filter invalid data
def filter_invalid_data(df):
    """Filter out rows with NaN, empty values, -2222, or -9999 in model_columns"""
    if df.shape[0] == 0:
        return df
    
    mask = True
    for col in model_columns:
        if col in df.columns:
            col_mask = (~df[col].isna()) & (df[col] != -2222) & (df[col] != -9999) & (df[col] != '')
            mask = mask & col_mask
        else:
            print(f"Warning: Column '{col}' not found in dataset")
    return df[mask].copy()

print(f"\n" + "="*40)
print("FILTERING DATA")
print("="*40)

print(f"Before filtering:")
print(f"  Train: {train_df.shape[0]:,} rows")
print(f"  Test: {test_df.shape[0]:,} rows")
print(f"  Evaluation: {evaluation_df.shape[0]:,} rows")

train_df_filtered = filter_invalid_data(train_df)
test_df_filtered = filter_invalid_data(test_df)
evaluation_df_filtered = filter_invalid_data(evaluation_df)

print(f"\nAfter filtering:")
print(f"  Train: {train_df_filtered.shape[0]:,} rows")
print(f"  Test: {test_df_filtered.shape[0]:,} rows")
print(f"  Evaluation: {evaluation_df_filtered.shape[0]:,} rows")

print(f"\nRows removed:")
print(f"  Train: {train_df.shape[0] - train_df_filtered.shape[0]:,}")
print(f"  Test: {test_df.shape[0] - test_df_filtered.shape[0]:,}")
print(f"  Evaluation: {evaluation_df.shape[0] - evaluation_df_filtered.shape[0]:,}")

# Calculate final percentages
total_filtered_rows = train_df_filtered.shape[0] + test_df_filtered.shape[0] + evaluation_df_filtered.shape[0]

if total_filtered_rows > 0:
    train_pct = (train_df_filtered.shape[0] / total_filtered_rows) * 100
    test_pct = (test_df_filtered.shape[0] / total_filtered_rows) * 100
    eval_pct = (evaluation_df_filtered.shape[0] / total_filtered_rows) * 100
    
    print(f"\nFinal split percentages:")
    print(f"  Train: {train_df_filtered.shape[0]:,} rows ({train_pct:.1f}%)")
    print(f"  Test: {test_df_filtered.shape[0]:,} rows ({test_pct:.1f}%)")
    print(f"  Evaluation: {evaluation_df_filtered.shape[0]:,} rows ({eval_pct:.1f}%)")
else:
    print(f"\nERROR: No rows remaining after filtering!")

# Verify no overlap between splits
print(f"\n" + "="*40)
print("OVERLAP VERIFICATION")
print("="*40)

def check_data_overlap(df1, df2, name1, name2):
    """Check for actual data overlap (not just campaigns)"""
    if df1.shape[0] > 0 and df2.shape[0] > 0:
        # Check if any rows are identical (using index)
        overlap_indices = set(df1.index) & set(df2.index)
        if overlap_indices:
            print(f"ERROR: Data overlap between {name1} and {name2}: {len(overlap_indices)} rows")
        else:
            print(f"✓ No data overlap between {name1} and {name2}")

check_data_overlap(train_df_filtered, test_df_filtered, "train", "test")
check_data_overlap(train_df_filtered, evaluation_df_filtered, "train", "evaluation")
check_data_overlap(test_df_filtered, evaluation_df_filtered, "test", "evaluation")

# Check campaign overlaps (expected for random split)
def check_campaign_overlap(df1, df2, name1, name2):
    if df1.shape[0] > 0 and df2.shape[0] > 0:
        campaigns1 = set(df1['Campaign'].unique())
        campaigns2 = set(df2['Campaign'].unique())
        overlap = campaigns1 & campaigns2
        if overlap:
            print(f"INFO: Campaign overlap between {name1} and {name2}: {overlap} (expected for random split)")
        else:
            print(f"INFO: No campaign overlap between {name1} and {name2}")

check_campaign_overlap(train_df_filtered, test_df_filtered, "train", "test")

# Save results
train_path = os.path.join(FINAL_OUT_PATH, "train.csv")
test_path = os.path.join(FINAL_OUT_PATH, "test.csv")
evaluation_path = os.path.join(FINAL_OUT_PATH, "evaluation.csv")

train_df_filtered.to_csv(train_path, index=False)
test_df_filtered.to_csv(test_path, index=False)
evaluation_df_filtered.to_csv(evaluation_path, index=False)

print(f"\nSaved files:")
print(f"  train.csv: {train_path}")
print(f"  test.csv: {test_path}")
print(f"  evaluation.csv: {evaluation_path}")

# Print final summary by campaign
print(f"\n" + "="*60)
print("FINAL SUMMARY BY CAMPAIGN")
print("="*60)

all_campaigns = set(df['Campaign'].unique())
for campaign in sorted(all_campaigns):
    train_count = train_df_filtered[train_df_filtered['Campaign'] == campaign].shape[0] if train_df_filtered.shape[0] > 0 else 0
    test_count = test_df_filtered[test_df_filtered['Campaign'] == campaign].shape[0] if test_df_filtered.shape[0] > 0 else 0
    eval_count = evaluation_df_filtered[evaluation_df_filtered['Campaign'] == campaign].shape[0] if evaluation_df_filtered.shape[0] > 0 else 0
    total_count = train_count + test_count + eval_count
    
    if total_count > 0:
        status = "HELD-OUT" if campaign in held_out_campaigns else "RANDOM-SPLIT"
        train_pct = (train_count / total_count * 100) if total_count > 0 else 0
        test_pct = (test_count / total_count * 100) if total_count > 0 else 0
        eval_pct = (eval_count / total_count * 100) if total_count > 0 else 0
        
        print(f"{campaign:20s} ({status:11s}): Train={train_count:6,} ({train_pct:4.1f}%), Test={test_count:6,} ({test_pct:4.1f}%), Eval={eval_count:6,} ({eval_pct:4.1f}%), Total={total_count:6,}")

print(f"\n" + "="*60)
print("RANDOM-BASED SPLIT COMPLETED")
print("="*60)
print(f"✅ Held-out campaigns: {len(held_out_campaigns)} campaigns → evaluation.csv")
print(f"✅ Random 80:20 split: {len(remaining_campaigns)} campaigns → train.csv & test.csv")
print(f"✅ All data filtered and saved successfully")
print(f"✅ Campaign mixing achieved: Same campaigns appear in both train and test")


METHOD 4: Random-Based Split (80:20 Train/Test + Hold-out Evaluation)
Reading random-based split configuration...
Random-based info shape: (1, 1)
Columns: ['held_out']
Held-out campaigns (go to evaluation): ['TCAP2013', 'DISCOVERAQ-California']
Evaluation data (held-out campaigns): 12,728 rows
Remaining data (for random split): 180,299 rows
Remaining campaigns for random split: ['ACMEV', 'ASIA-AQ', 'BBOP', 'CACTI', 'CARES', 'DC3', 'DISCOVERAQ-Texas', 'FIREXAQ', 'NAAMES(2015)', 'NAAMES(2016)', 'NAAMES(2017)']

Performing random 80:20 split on remaining 180,299 rows...
Random split results:
  Train: 144,239 rows (80.0%)
  Test: 36,060 rows (20.0%)

Campaign distribution in random splits:
Train campaigns:
  ASIA-AQ: 108,678 rows (75.3%)
  FIREXAQ: 29,869 rows (20.7%)
  BBOP: 3,640 rows (2.5%)
  DISCOVERAQ-Texas: 657 rows (0.5%)
  DC3: 525 rows (0.4%)
  ACMEV: 412 rows (0.3%)
  CACTI: 301 rows (0.2%)
  CARES: 143 rows (0.1%)
  NAAMES(2017): 13 rows (0.0%)
  NAAMES(2015): 1 rows (0.0%)
Tes